# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

# TF1 环境下默认不是 eager，必须显式开启（必须在任何 TF op 之前调用）
try:
    tf.compat.v1.enable_eager_execution()
except Exception:
    pass

print("eager:", tf.executing_eagerly())
print("tf version:", tf.__version__)

eager: True
tf version: 1.15.0


## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [2]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['FZOZVYSKZY', 'LDUSPNICGH'], <tf.Tensor: id=0, shape=(2, 10), dtype=int32, numpy=
array([[ 6, 26, 15, 26, 22, 25, 19, 11, 26, 25],
       [12,  4, 21, 19, 16, 14,  9,  3,  7,  8]])>, <tf.Tensor: id=1, shape=(2, 10), dtype=int32, numpy=
array([[ 0, 25, 26, 11, 19, 25, 22, 26, 15, 26],
       [ 0,  8,  7,  3,  9, 14, 16, 19, 21,  4]])>, <tf.Tensor: id=2, shape=(2, 10), dtype=int32, numpy=
array([[25, 26, 11, 19, 25, 22, 26, 15, 26,  6],
       [ 8,  7,  3,  9, 14, 16, 19, 21,  4, 12]])>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [3]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz = 27
        self.hidden = 128

        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)

        # 只用 Cell，不用 tf.keras.layers.RNN（避免触发 estimator 相关依赖）
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)

        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)

    def _run_rnn_cell(self, cell, inputs, initial_state):
        """
        inputs: [B, T, E]
        initial_state: [B, H]
        return:
          outs: [B, T, H]
          last_state: [B, H]
        """
        # eager 下尽量用静态长度；如果没有就退化到 numpy() 取值
        T = inputs.shape[1]
        if T is None:
            T = int(tf.shape(inputs)[1].numpy())

        state = initial_state
        outs = []
        for t in range(T):
            x_t = inputs[:, t, :]              # [B, E]
            out_t, [state] = cell(x_t, [state])  # out_t [B,H], state [B,H]
            outs.append(out_t)
        outs = tf.stack(outs, axis=1)          # [B,T,H]
        return outs, state

    @tf.autograph.experimental.do_not_convert
    def call(self, enc_ids, dec_ids):
        # encode
        enc_emb = self.embed_layer(enc_ids)  # [B,S,E]
        bsz = tf.shape(enc_emb)[0]
        enc_h0 = tf.zeros(tf.stack([bsz, self.hidden]), dtype=enc_emb.dtype)  # 关键：用 tf.stack
        enc_out, enc_state = self._run_rnn_cell(self.encoder_cell, enc_emb, enc_h0)  # [B,S,H], [B,H]

        # decode (teacher forcing)
        dec_emb = self.embed_layer(dec_ids)  # [B,T,E]
        dec_out, _ = self._run_rnn_cell(self.decoder_cell, dec_emb, enc_state)  # [B,T,H], [B,H]

        # attention
        enc_out_proj = self.dense_attn(enc_out)                              # [B,S,H]
        scores = tf.einsum("bth,bsh->bts", dec_out, enc_out_proj)            # [B,T,S]
        alpha = tf.nn.softmax(scores, axis=-1)                               # [B,T,S]
        context = tf.einsum("bts,bsh->bth", alpha, enc_out)                  # [B,T,H]

        fused = dec_out + self.dense_attn(context)                           # [B,T,H]
        logits = self.dense(fused)                                           # [B,T,V]
        return logits

    @tf.autograph.experimental.do_not_convert
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids)  # [B,S,E]
        bsz = tf.shape(enc_emb)[0]
        enc_h0 = tf.zeros(tf.stack([bsz, self.hidden]), dtype=enc_emb.dtype)  # 关键：用 tf.stack
        enc_out, enc_state = self._run_rnn_cell(self.encoder_cell, enc_emb, enc_h0)
        return enc_out, [enc_out[:, -1, :], enc_state]

    def get_next_token(self, x, state, enc_out):
        prev_out, dec_state = state[0], state[1]

        # 1-step decode
        x_emb = self.embed_layer(tf.expand_dims(x, axis=1))  # [B,1,E]
        x_emb = tf.squeeze(x_emb, axis=1)                    # [B,E]
        dec_out, [new_dec_state] = self.decoder_cell(x_emb, [dec_state])  # [B,H]

        # attention
        enc_out_proj = self.dense_attn(enc_out)              # [B,S,H]
        scores = tf.einsum("bh,bsh->bs", dec_out, enc_out_proj)  # [B,S]
        alpha = tf.nn.softmax(scores, axis=-1)               # [B,S]
        context = tf.einsum("bs,bsh->bh", alpha, enc_out)    # [B,H]

        fused = dec_out + self.dense_attn(context)           # [B,H]
        logits = self.dense(fused)                           # [B,V]
        out = tf.argmax(logits, axis=-1, output_type=tf.int32)

        new_state = [fused, new_dec_state]
        return out, new_state

# Loss函数以及训练逻辑

In [4]:
# 删掉/注释掉 @tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(logits=logits, labels=labels)
    return tf.reduce_mean(losses)

# 删掉/注释掉 @tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            loss_val = loss.numpy() if hasattr(loss, "numpy") else loss
            print('step', step, ': loss', loss_val)
    return loss

# 训练迭代

In [5]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.314878
step 500 : loss 0.65736365
step 1000 : loss 0.09328567
step 1500 : loss 0.04827749


<tf.Tensor: id=2297800, shape=(), dtype=float32, numpy=0.034542605>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [7]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, False, False, True, True, True, False, False, True, True, True, True, False, False, False, True, True, True, True, True, True, False, True, False, True, True, False, True, True, True, False]
[('SAAFDSGCLOSPHDAWCBJI', 'IJBCWADHPSOLCGSDFAAE'), ('PBXQTCIKTEYQXRLDDVFU', 'UFVDDLRXQYETKICTQXBP'), ('HIVCHKWCCDQHQDCVJWJP', 'PJWJVCDQHQDCCWKHCVIB'), ('JEBJJMBRAGYLUVXMLWJF', 'FJWLMXVULYGARBMJJBEZ'), ('YUZVQIENIVCIVYSNMCDX', 'XDCMNSYVICVINEIQVZUY'), ('CBYIZUCFLOKVDQSNEGZU', 'UZGENSQDVKOLFCUZIYBC'), ('TNXSSAQTUEPNPFXTTVJO', 'OJVTTXFPNPEUTQASSXNT'), ('MRKGFBMWCFWDCJANKRNK', 'KNRKNAJCDWFCWMBFGKRM'), ('ETECWIQQGEOAVZLGTXCY', 'YCXTGLZVAOEGQQIWCETE'), ('FKILOPLEJSXSNIIVYVAK', 'KAVYVIINSXSJELPOLIKF'), ('BCTAOBLXYIPSLXXEHXWI', 'IWXHEXXLSPIYXLBOATCB'), ('RKAERYQYPTQLTZNATYYD', 'DYYTANZTLQTPYQYREAKR'), ('QHLHFOKDGKILQGCXWZXD', 'DXZWXCGQLIKGDKOFHLHQ'), ('UNVBEVWFQQINUBUQTPVE', 'EVPTQUBUNIQQFWVEBVNU'), ('LIBQXHYUNLFKLBUXQEIL', 'LIEQXUBLKFLNUYHXQBIL'), ('CEKJMOMMEANYVIOSYPFK', 'KFPYSOIVYNAEMMOMJKE